In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

spark.sql("""
CREATE OR REPLACE VIEW main.gold.v_time_series_region AS
SELECT
  date_trunc('day', date_prelevement) AS jour,
  parametre,
  AVG(resultat_numerique) AS moyenne,
  MIN(resultat_numerique) AS min_val,
  MAX(resultat_numerique) AS max_val,
  COUNT(*) AS nb_mesures
FROM main.silver.qualite_eau_region
WHERE resultat_numerique IS NOT NULL
GROUP BY date_trunc('day', date_prelevement), parametre
""")

spark.sql("""
CREATE OR REPLACE VIEW main.gold.v_parametre_stats_region AS
SELECT
  parametre,
  COUNT(*) AS nb_mesures,
  AVG(resultat_numerique) AS moyenne,
  MIN(resultat_numerique) AS min_val,
  MAX(resultat_numerique) AS max_val
FROM main.silver.qualite_eau_region
WHERE resultat_numerique IS NOT NULL
GROUP BY parametre
""")

spark.sql("""
CREATE OR REPLACE VIEW main.gold.v_reseau_quality_region AS
SELECT
  reseau_nom,
  COUNT(*) AS nb_mesures,
  SUM(CASE WHEN upper(conformite_limites_pc_prelevement) = 'NC' THEN 1 ELSE 0 END) AS nc_pc,
  SUM(CASE WHEN upper(conformite_limites_bact_prelevement) = 'NC' THEN 1 ELSE 0 END) AS nc_bact
FROM main.silver.qualite_eau_region
GROUP BY reseau_nom
""")

spark.sql("""
CREATE OR REPLACE VIEW main.gold.v_depassements_region AS
WITH base AS (
  SELECT
    date_prelevement,
    commune,
    parametre,
    resultat_numerique,

    try_cast(
      regexp_replace(
        regexp_replace(limite_qualite_parametre, ',', '.'),
        '[^0-9.]',
        ''
      ) AS double
    ) AS limite_numerique

  FROM main.silver.qualite_eau_region
  WHERE resultat_numerique IS NOT NULL
)

SELECT
  date_prelevement,
  commune,
  parametre,
  resultat_numerique,
  limite_numerique,
  CASE
    WHEN limite_numerique IS NOT NULL
     AND resultat_numerique > limite_numerique
    THEN 1 ELSE 0
  END AS depassement
FROM base
""")

spark.sql("""
CREATE OR REPLACE VIEW main.gold.v_commune_quality_region AS
SELECT
  commune,
  COUNT(*) AS nb_mesures,
  AVG(resultat_numerique) AS moyenne,
  MIN(resultat_numerique) AS min_val,
  MAX(resultat_numerique) AS max_val
FROM main.silver.qualite_eau_region
GROUP BY commune
""")

spark.sql("""
CREATE OR REPLACE VIEW main.gold.v_top_anomalies_region AS
SELECT
  date_prelevement,
  commune,
  parametre,
  resultat_numerique,
  limite_qualite_parametre
FROM main.silver.qualite_eau_region
WHERE resultat_numerique IS NOT NULL
ORDER BY resultat_numerique DESC
""")